# PyPer - PR Times to Gmail Pipeline

PR Times からプレスリリースを取得して Gmail で送信するパイプラインです。

## 手順

1. 以下のセルを実行して依存関係をインストール
2. Google 認証を実行
3. パイプラインを実行

In [ ]:
# 依存関係のインストール
!pip install -q pyyaml requests beautifulsoup4 python-dotenv google-auth google-auth-oauthlib google-auth-httplib2

In [ ]:
# Google 認証
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
import os
import json

# 認証スコープ
SCOPES = ['https://mail.google.com/']

# 認証実行
flow = InstalledAppFlow.from_client_config(
    {
        "installed": {
            "client_id": "YOUR_CLIENT_ID",
            "client_secret": "YOUR_CLIENT_SECRET",
            "auth_uri": "https://accounts.google.com/o/oauth2/auth",
            "token_uri": "https://oauth2.googleapis.com/token",
            "redirect_uris": ["https://localhost"]
        }
    },
    SCOPES
)

print("Google 認証 URL を開いてください...")
creds = flow.run_local_server(port=0, open_browser=False)

# トークンを保存（後で使用）
with open('oauth_token.json', 'w') as f:
    json.dump({
        'access_token': creds.token,
        'refresh_token': creds.refresh_token,
        'token_uri': creds.token_uri,
        'client_id': creds.client_id,
        'client_secret': creds.client_secret,
    }, f)

print("✓ 認証完了！")
print(f"リフレッシュトークン：{creds.refresh_token}")

In [ ]:
# パイプライン実行
import sys
sys.path.insert(0, '/content/PyPer/src')

# GitHub からコードをクローン（初回のみ）
if not os.path.exists('/content/PyPer'):
    !git clone https://github.com/bonsai/PyPer.git

from main import run_pipeline

# 設定
RECIPIENT_EMAIL = "your_email@gmail.com"  # 送信先メールアドレス

# パイプライン実行
print("パイプラインを実行中...")
run_pipeline('/content/PyPer/recipe/prt.yaml')